# v40d — CMB m-Parity Test: Planck 2018 SEVEM

**KTF³ Analysis — Andrew Cotting, April 2026**

## Hypothesis

KTF³ non-orientable topology predicts: odd-m CMB modes carry more power than even-m modes.
Statistic: R = (P_odd - P_even) / (P_odd + P_even) > 0

## Cross-instrument series

| Notebook | Map | σ |
|----------|-----|---|
| v40b | SMICA | +0.46 |
| v40c | NILC | +1.14 |
| v40d | SEVEM | +0.48 |
| v40e | COMMANDER | +0.37 |
| v40f | WMAP 9yr | +1.06 |
| v40g | ACT DR4 | pending |

**This notebook: SEVEM**

## Data

File: `COM_CMB_IQU-sevem_2048_R3.00_full.fits`
Download: IRSA Planck → release_3 → all-sky-maps → SEVEM
URL: `https://irsa.ipac.caltech.edu/data/Planck/release_3/all-sky-maps/previews/COM_CMB_IQU-sevem_2048_R3.00_full/COM_CMB_IQU-sevem_2048_R3.00_full.fits`

Upload to Colab before running.

In [ ]:
!pip install healpy numpy scipy matplotlib astropy -q
print('Packages ready.')

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

MAPNAME = 'SEVEM'
VNAME   = 'v40d'
print(f'{VNAME} — m-Parity Test: Planck 2018 {MAPNAME}')
print('KTF³: non-orientable → odd-m > even-m (R > 0)')
print('=' * 50)

In [ ]:
# === LOAD MAP ===
# Upload the FITS file to Colab before running this cell.
# File: COM_CMB_IQU-sevem_2048_R3.00_full.fits
# URL:  https://irsa.ipac.caltech.edu/data/Planck/release_3/all-sky-maps/previews/COM_CMB_IQU-sevem_2048_R3.00_full/COM_CMB_IQU-sevem_2048_R3.00_full.fits

FITS_FILE = 'COM_CMB_IQU-sevem_2048_R3.00_full.fits'

# Auto-detect filename variants
if not os.path.exists(FITS_FILE):
    alt = FITS_FILE.replace('.fits', '-nosz_2048_R3.00_full.fits').replace('_2048_R3.00_full', '-nosz_2048_R3.00_full')
    for f in os.listdir('.'):
        if 'sevem' in f.lower() and f.endswith('.fits'):
            os.rename(f, FITS_FILE)
            print(f'Renamed: {f} -> ' + FITS_FILE)
            break

if not os.path.exists(FITS_FILE):
    raise FileNotFoundError(
        f'File not found: {FITS_FILE}\n'
        f'Download from: https://irsa.ipac.caltech.edu/data/Planck/release_3/all-sky-maps/previews/COM_CMB_IQU-sevem_2048_R3.00_full/COM_CMB_IQU-sevem_2048_R3.00_full.fits\n'
        f'Upload to Colab, then re-run.'
    )

print(f'Loading: {FITS_FILE}')
print(f'Size: {os.path.getsize(FITS_FILE)/1e6:.0f} MB')

# Read T map (field 0)
raw = hp.read_map(FITS_FILE, field=0, verbose=False)
NSIDE_RAW = hp.get_nside(raw)
print(f'NSIDE: {NSIDE_RAW}')
print(f'Map range: {raw[raw!=hp.UNSEEN].min():.1f} to {raw[raw!=hp.UNSEEN].max():.1f}')

In [ ]:
# === PREPROCESS ===
NSIDE = 128
LMAX  = 256

# Downgrade
if NSIDE_RAW != NSIDE:
    cmb = hp.ud_grade(raw, NSIDE)
else:
    cmb = raw.copy()

# Mask: galactic plane |b| > 20°
npix  = hp.nside2npix(NSIDE)
theta, phi = hp.pix2ang(NSIDE, np.arange(npix))
b_deg = 90.0 - np.degrees(theta)
mask  = (np.abs(b_deg) > 20.0) & (cmb != hp.UNSEEN)

cmb_masked = cmb.copy()
cmb_masked[~mask] = hp.UNSEEN

f_sky = mask.mean()
print(f'Mask: |b| > 20°, f_sky = {f_sky:.3f}')
print(f'Pixels retained: {mask.sum()}')

In [ ]:
# === m-PARITY STATISTIC ===
def mparity(cmb_map, lmax):
    alm    = hp.map2alm(cmb_map, lmax=lmax)
    P_odd  = 0.0
    P_even = 0.0
    for l in range(2, lmax + 1):
        for m in range(0, l + 1):
            power = np.abs(alm[hp.Alm.getidx(lmax, l, m)]) ** 2
            if m % 2 == 0:
                P_even += power
            else:
                P_odd  += power
    total = P_odd + P_even
    return (P_odd - P_even) / total if total > 0 else 0.0

print('Computing m-parity...')
R_obs = mparity(cmb_masked, LMAX)
print(f'R_obs = {R_obs:.6f}')
print('(Positive = odd-m dominant = consistent with KTF³)')

In [ ]:
# === ΛCDM NULL SIMULATIONS ===
N_SIM = 500
print(f'Running {N_SIM} ΛCDM simulations (~5-10 min)...')

cl_data = hp.anafast(cmb_masked, lmax=LMAX)
cl_fid  = np.maximum(cl_data / f_sky, 0)

R_sim = np.zeros(N_SIM)
for i in range(N_SIM):
    sim = hp.synfast(cl_fid, nside=NSIDE, lmax=LMAX, verbose=False, new=True)
    sim[~mask] = hp.UNSEEN
    R_sim[i] = mparity(sim, LMAX)
    if (i+1) % 100 == 0:
        print(f'  {i+1}/{N_SIM}')

sigma = (R_obs - R_sim.mean()) / R_sim.std()
p_val = np.mean(R_sim >= R_obs)

print(f'\nR_obs  = {R_obs:.6f}')
print(f'R_sim  = {R_sim.mean():.6f} ± {R_sim.std():.6f}')
print(f'σ      = {sigma:.3f}')
print(f'p      = {p_val:.4f}')

In [ ]:
# === PLOT ===
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('v40d — m-Parity: Planck SEVEM (Cotting 2026)', fontsize=12, fontweight='bold')

# Panel 1: MC null
ax = axes[0]
ax.hist(R_sim, bins=40, color='steelblue', edgecolor='white', alpha=0.85,
        density=True, label='ΛCDM null (N=500)')
ax.axvline(R_obs, color='red', lw=2.5, label=f'R_obs = {R_obs:.4f}')
ax.axvline(R_sim.mean(), color='gray', lw=1.5, ls='--', label='Sim mean')
ax.set_xlabel('m-parity statistic R')
ax.set_ylabel('Density')
ax.set_title(f'σ = {sigma:.2f}   p = {p_val:.4f}')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: Cross-instrument
ax = axes[1]
instruments = ['SMICA', 'NILC', 'SEVEM', 'CMD', 'WMAP']
sigmas_all  = [0.46, 1.14, 0.48, 0.37, 1.06]
# Replace this map's known value with freshly computed sigma
idx = 2
sigmas_all[idx] = sigma
colors = ['#4477AA']*5
colors[idx] = '#EE6677'
bars = ax.bar(instruments, sigmas_all, color=colors, edgecolor='white', alpha=0.9)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(2, color='orange', lw=1.5, ls='--', label='2σ')
for bar, s in zip(bars, sigmas_all):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
            f'{s:.2f}', ha='center', fontsize=9)
ax.set_ylabel('σ (m-parity)')
ax.set_title('Cross-Instrument Comparison')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('v40d_mparity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v40d_mparity.png')

In [ ]:
# === SUMMARY ===
print('=' * 55)
print('v40d — m-Parity Test: Planck SEVEM')
print('Cotting, S. (2026) — KTF³ Analysis')
print('=' * 55)
print(f'Map:     SEVEM')
print(f'NSIDE:   {NSIDE}')
print(f'lmax:    {LMAX}')
print(f'f_sky:   {f_sky:.3f}')
print(f'N_SIM:   {N_SIM}')
print()
print(f'R_obs:   {R_obs:.6f}')
print(f'R_sim:   {R_sim.mean():.6f} ± {R_sim.std():.6f}')
print(f'σ:       {sigma:.3f}')
print(f'p:       {p_val:.4f}')
print()
print('Cross-instrument (all positive → not artefact):')
print('  SMICA:     σ = +0.46')
print('  NILC:      σ = +1.14')
print('  SEVEM:     σ = +0.48')
print('  COMMANDER: σ = +0.37')
print('  WMAP 9yr:  σ = +1.06')
print(f'  {MAPNAME:10s}: σ = {sigma:+.2f}  ← this result')
print()
print('GitHub: github.com/Andrew-Cot/KTF3-Analyse')
print('Pre-reg: academia.edu/AndrewCotting')
print('=' * 55)